# E-Gadget Addiction — Exploratory Data Analysis
Run cells top to bottom. Make sure `student_data.csv` exists in `data/`.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/student_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic info
df.info()
df.describe()

In [ ]:
# Class distribution
order = ['Low', 'Moderate', 'High', 'Severe']
colors = ['#27ae60', '#f39c12', '#e67e22', '#c0392b']
counts = df['addiction_label'].value_counts().reindex(order)
counts.plot(kind='bar', color=colors, edgecolor='white', rot=0)
plt.title('Addiction Risk Level Distribution')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include='number')
plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Screen time vs risk level
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x='addiction_label', y='daily_screen_time_hours',
            order=order, palette=colors, ax=axes[0])
axes[0].set_title('Screen Time by Risk Level')

sns.boxplot(data=df, x='addiction_label', y='sleep_hours',
            order=order, palette=colors, ax=axes[1])
axes[1].set_title('Sleep Hours by Risk Level')

plt.tight_layout()
plt.show()

In [ ]:
# GPA vs stress
plt.figure(figsize=(9, 5))
for label, color in zip(order, colors):
    subset = df[df['addiction_label'] == label]
    plt.scatter(subset['gpa'], subset['stress_level'],
                label=label, color=color, alpha=0.6, s=40)
plt.xlabel('GPA')
plt.ylabel('Stress Level')
plt.title('GPA vs Stress Level by Risk Category')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Train model and show feature importance
from train_model import train
import joblib

rf, feature_names = train()

importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(feature_names)),
        importances[idx], color='#3498db', edgecolor='white')
plt.xticks(range(len(feature_names)),
           [feature_names[i] for i in idx], rotation=45, ha='right')
plt.title('Random Forest — Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP global importance
from preprocess import preprocess
from shap_explainer import plot_global_importance

X, y, feat_names = preprocess(apply_smote=False)
plot_global_importance(X[:200], feat_names, rf, save_path='../reports/shap_global.png')

from IPython.display import Image
Image('../reports/shap_global.png')